# Datasplit via tanimoto similarity

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import numpy as np


def compute_similarity_matrix(molecules, fp_function=compute_countMorgFP):  #A square matrix of similarity between all inputted molecules. Input can be in mol or smiles format.
    fps = fp_function(molecules)
    
    similarity_matrix = np.zeros((len(fps), len(fps)))
    for i in range(len(fps)):
        for j in range(i, len(fps)):
            similarity = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            similarity_matrix[i][j] = similarity_matrix[j][i] = similarity
    return similarity_matrix


def initial_iteration(molecules, similarity_matrix, similarity_matrix_MS, cutoff, cutoff_MS, filter_by_AnonMS = True):
    transfer_count = {}
    avg_MS_similarity = {}

    AnonMS_dict = {}
    for i in range(len(molecules)):
        smi = Chem.MolToSmiles(molecules[i])
        AnonMS_dict[i] = get_anonymous_murcko(smi)

    for i in tqdm(list(range(0, len(molecules)))):
        if i in transfer_count: #dont recompute what is known
            continue

        #get transfers
        test_set = set([i])
        training_set = set(range(len(molecules))) - test_set

        test_smi = Chem.MolToSmiles(molecules[i])
        test_AnonMS_set = set([get_anonymous_murcko(test_smi)])

        train_AnonMS_dict = dict(AnonMS_dict)
        del train_AnonMS_dict[i]
        

        transfers = build_test_set(test_set, training_set, similarity_matrix, similarity_matrix_MS, cutoff, cutoff_MS, molecules, filter_by_AnonMS)
        transfer_count[i] = len(transfers)
        
        #Calculate average MS similarity between resulting test set and training set for each given seed molecule. As to select the lowest avg MS similarity later
        resulting_test_set = test_set.union(transfers)
        resulting_training_set = set(range(len(molecules))) - resulting_test_set
        sum_MS_similarity = 0
        for test_idx in resulting_test_set:
            for train_idx in resulting_training_set:
                sum_MS_similarity += similarity_matrix_MS[train_idx][test_idx]
        if len(resulting_test_set)*len(resulting_training_set)!=0:
            avg_MS_similarity[i] = sum_MS_similarity/(len(resulting_test_set)*len(resulting_training_set))
        else:
            avg_MS_similarity[i] = 1

        #assign the same transfer_count and avg_MS_similarity to all molecules/idx which were transferred
        for idx in transfers:
            transfer_count[idx] = transfer_count[i]
            avg_MS_similarity[idx] = avg_MS_similarity[i]


    return transfer_count, avg_MS_similarity

def build_test_set(test_set, training_set, similarity_matrix, similarity_matrix_MS, cutoff, cutoff_MS, molecules, filter_by_AnonMS = True):
    transfers = set() #Set of idx transfered from training set to test set
    changed = True
   

    smiles_anon_ms_list = [get_anonymous_murcko(Chem.MolToSmiles(mol)) for mol in molecules] #Smiles format of all AnonMS in order of the molecules. Repeats do of AnonMS occur as multiple mol share AnonMS.
    #AnonMS_set = set(smiles_anon_ms_list)
    test_AnonMS_list_2 = [smiles_anon_ms_list[test_idx] for test_idx in test_set] #smiles_anon_ms_list[list(test_set)]  
    test_AnonMS_set_2 = set(test_AnonMS_list_2)
    #train_AnonMS_list_2 = smiles_anon_ms_list[list(training_set)]
    #train_AnonMS_set_2 = set(train_AnonMS_list_2)



    while changed:
        changed = False
        for train_idx in list(training_set):
            for test_idx in list(test_set):

                shall_transfer = False 
                if smiles_anon_ms_list[train_idx] in test_AnonMS_set_2 and filter_by_AnonMS==True: #if AnonMS of train mol is in testset
                    shall_transfer = True

                elif (similarity_matrix[train_idx][test_idx] > cutoff) or (similarity_matrix_MS[train_idx][test_idx] > cutoff_MS): #or general MS of Train idx is in Test set
                    shall_transfer = True
                    if smiles_anon_ms_list[train_idx] not in test_AnonMS_set_2:
                        test_AnonMS_set_2.add(smiles_anon_ms_list[train_idx])
                    
                if shall_transfer:
                    test_set.add(train_idx)
                    transfers.add(train_idx)
                    training_set.remove(train_idx)        
                    changed = True 
                    break    
                        
    return transfers



def build_final_test_set(molecules, similarity_matrix, similarity_matrix_MS, transfer_count, avg_MS_similarity, test_size_ratio, cutoff, cutoff_MS, filter_by_AnonMS = True):
    
    #transfer_count = { ... } # Your dictionary with molecule index as keys and number of transfers as values
    #avg_MS_Similarity = [...] # Your avg_MS_Similarity list

    # Create a list of tuples (index, transfer_count, avg_similarity) 
    indexed_transfer_count = [(index, count, avg_MS_similarity[index]) for index, count in transfer_count.items()]

    # Sort the list by transfer_count primarily, and by avg_similarity in ascending order secondarily
    sorted_molecules_tuple = sorted(indexed_transfer_count, key=lambda x: (x[1], x[2]))

    # Extract only the molecule indexes from the sorted list
    sorted_molecule_indexes = [index for index, _, _ in sorted_molecules_tuple]


    test_set = set()
    training_set = set(range(len(molecules)))


    num_molecular_groups = 0

    while len(test_set) < int(len(molecules) * test_size_ratio):
        for idx in sorted_molecule_indexes:
            if idx not in test_set:
                prev_test_set = test_set.copy()
                prev_training_set = training_set.copy()
                test_set.add(idx)
                training_set.remove(idx)

                build_test_set(test_set, training_set, similarity_matrix, similarity_matrix_MS, cutoff, cutoff_MS, molecules, filter_by_AnonMS)
                num_molecular_groups += 1
                if len(test_set) >= int(len(molecules) * test_size_ratio):
                    break
 
                
    if len(test_set) >= int(len(molecules) * test_size_ratio * 1.2): #Allow for the addition of the final group to be added if it doesnt increase the testset too much
        print(f'Number of seed molecules: {num_molecular_groups-1}')
        final_test_set = [molecules[i] for i in prev_test_set]
        final_training_set = [molecules[i] for i in prev_training_set]
        print(f'Final addition was {Chem.MolToSmiles(molecules[idx])} and it made the test set increase from size {len(prev_test_set)}, to {len(test_set)}. So this was undone and it was not added.')
    else:
        print(f'Number of seed molecules: {num_molecular_groups}')
        final_test_set = [molecules[i] for i in test_set]
        final_training_set = [molecules[i] for i in training_set]
    return final_test_set, final_training_set



def generate_test_train_split(unique_substructures, cutoff, cutoff_MS, max_test_fraction, filter_by_AnonMS = True, fp_function = compute_countMorgFP):
    molecules = [Chem.MolFromSmiles(smiles) for smiles in unique_substructures]
    molecules_MS = [get_mol(get_murcko(smiles)) for smiles in unique_substructures]
    similarity_matrix = compute_similarity_matrix(molecules, fp_function=fp_function)
    similarity_matrix_MS = compute_similarity_matrix(molecules_MS, fp_function=fp_function)
    transfer_count, avg_MS_similarity = initial_iteration(molecules, similarity_matrix, similarity_matrix_MS, cutoff, cutoff_MS, filter_by_AnonMS)
    final_test_set, final_training_set = build_final_test_set(molecules, similarity_matrix, similarity_matrix_MS, transfer_count, avg_MS_similarity, max_test_fraction, cutoff, cutoff_MS, filter_by_AnonMS)
    print(f'Final number of test molecules: {len(final_test_set)} out of {len(unique_substructures)}')

    
    final_test_smi_set = [Chem.MolToSmiles(mol) for mol in final_test_set]
    final_training_smi_set = [Chem.MolToSmiles(mol) for mol in final_training_set]

    temp = []
    for smi in final_test_smi_set:
        smi_anon_ms = get_anonymous_murcko(smi)
        temp.append(smi_anon_ms)
    print(f'Number of general Murcko scaffolds {len(set(temp))}')

    return final_test_set, final_training_set, final_test_smi_set, final_training_smi_set




In [ ]:
from rdkit.Chem import rdFingerprintGenerator
def compute_RDKitFP(smiles, maxPath=7, fpSize=2048):
    if isinstance(smiles[0], str):
        mols = [get_mol(smi) for smi in smiles]
    else:
        mols = smiles #assume mols were fed instead
    rdgen = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=maxPath, fpSize=fpSize)
    fps = [rdgen.GetCountFingerprint(mol) for mol in mols]
    return fps

In [ ]:
from rdkit.Avalon.pyAvalonTools import GetAvalonCountFP, GetAvalonFP
def compute_AvalonFP(smiles):
    if isinstance(smiles[0], str):
        mols = [get_mol(smi) for smi in smiles] 
    else:
        mols = smiles #assume mols were fed instead
    fps = [GetAvalonCountFP(mol) for mol in mols]
    return fps

In [ ]:
max_test_fraction = 0.2
cutoff = 0.399
cutoff_MS = cutoff


poi_test_set_without_attachment, poi_trainval_set_without_attachment, poi_test_smi_set_without_attachment, poi_trainval_smi_set_without_attachment = generate_test_train_split(unique_substructures=unique_poi_without_attachment, cutoff=cutoff, cutoff_MS=cutoff_MS, max_test_fraction=max_test_fraction, filter_by_AnonMS=False, fp_function= compute_countMorgFP)

In [ ]:
#remove "linker" with no size, which directly joins POI and E3
while "[*:1][*:2]" in unique_linker:
    unique_linker.remove("[*:1][*:2]")
while "[*:2][*:1]" in unique_linker:
    unique_linker.remove("[*:2][*:1]")

while "[H][H]" in unique_linker_without_attachment:
    unique_linker_without_attachment.remove("[H][H]")

In [ ]:
max_test_fraction = 0.2  #0.5 is too high for RDKitFP and TopologicalTorsion, but lower values leads to smaller testset as the next addition would include the whole training set => Training cluster is smaller in RDkitFP and TTFP, but it is more diffuse.
cutoff = 0.449
cutoff_MS = cutoff #higher values tended to include smaller linkers with very small differences in both train and test set

linker_test_set_without_attachment, linker_trainval_set_without_attachment, linker_test_smi_set_without_attachment, linker_trainval_smi_set_without_attachment = generate_test_train_split(unique_substructures=unique_linker_without_attachment, cutoff=cutoff, cutoff_MS=cutoff_MS, max_test_fraction=max_test_fraction, filter_by_AnonMS=False, fp_function = compute_RDKitFP)

#370 SMILES, 213 frameworks at 0.5/0.5 RDKit fp - 15% points lower ratio of frameworks/smiles => SMILES with same frameworks are more similar in RDKitFP?
#68 SMILES,  52 frameworks at 0.5/0.5 countMorg fp. Was stopped since the next cluster encompassed the rest of the data.
#88 SMILES,  69 frameworks at 0.5/0.5 Morg fp
#327 SMILES, 188 frameworks at 0.5/0.5 RDKitFP with maxpath = 15 - 2 ish times slower
#118 SMILES, 70 frameworks at 0.5/0.5 TopologicalTorsion. Was stopped since the next cluster encompassed the rest of the data. Somehow the clustering failed??? similarity of 0.94
#164 SMILES, 94 frameworks at 0.5/0.5 RDKit fp, minPath=2, 3 times slower
#149 SMILES, 86 frameworks at 0.5/0.5 RDKit fp, minPath=2, not count based, 1.5 times slower
#93 SMILES, 69 frameworks at 0.5/0.5 MorgFP, radius=3. 1.5 times slower, fewer linker-frameworks that leak between test and train sets than RDKit and TopTorsFP.
#68 SMILES, 52 frameworks at 0.5/0.5 MorgFP, radius=2. reference speed, FEWEST linker-frameworks that leak between test and train sets than RDKit (standard settings) and TopTorsFP.
#8 SMILES, 8 frameworks at 0.45/0.45 MorgFP, radius=2. reference speed
#74 SMILES, 48 frameworks at 0.4/0.4 RDKitfp, reference speed
#104  SMILES, 72 frameworks at 0.45/0.45 RDKitfp, reference speed

#Want NumSeedMolecules/GeneralMurckoScaffolds=1 ish, and to maximize GeneralMurckoScaffolds/testMol and NumSeedMolecules/testMolecules, which still satesfies the set tanimito similarity cutoff (all results from generate_test_train_split will do so)
    #NumSeedMolecules & NumGeneralMurckoScaffolds are measures of diversity, but Seed molecules is tanimoto divesrity and GMS is framework-diversity, sort of.
    #Maximize NumSeedMolecules/GeneralMurckoScaffolds, since otherwise the chosen fingerprint will not be able to differentiate between molecules with different frameworks. If numSeedMolecules >> GeneralMurckoScaffolds, then there is low diversity 
    #Avalon:        NumSeedMolecules=4      GeneralMurckoScaffolds=144    testMolecules=258
    #CountAvalon:   NumSeedMolecules=21     GeneralMurckoScaffolds=14     testMolecules=22
    #Morgan:        NumSeedMolecules=8      GeneralMurckoScaffolds=8      testMolecules=8
    #CountRDkit:    NumSeedMolecules=32     GeneralMurckoScaffolds=72     testMolecules=104
    


In [ ]:
max_test_fraction = 0.2
cutoff = 0.399
cutoff_MS = cutoff

e3_test_set_without_attachment, e3_trainval_set_without_attachment, e3_test_smi_set_without_attachment, e3_trainval_smi_set_without_attachment = generate_test_train_split(unique_substructures=unique_e3_without_attachment, cutoff=cutoff, cutoff_MS=cutoff_MS, max_test_fraction=max_test_fraction, filter_by_AnonMS=False, fp_function = compute_countMorgFP)


In [ ]:
# len(unique_linker) == 1033 #with attachment
#len(unique_linker_without_attachment) = 944    #without
#generate_test_train_split(unique_linker_without_attachment) -> len(linker_trainval_smi_set_without_attachment) #without
#len(linker_trainval_smi_set_without_attachment) == 873             #without
#len(set(linker_trainval_smi_set_without_attachment))  == 873       #without
# len(set(linker_trainval_smi_set)) == len(linker_trainval_smi_set) == 961    #with attachment points
#Number of test Linker: 72 #with attachment points
# 961+72 (with attachment)
